In [35]:
import re
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML as IPHTML


In [36]:
APP_NAME = "DocLens"

LEGAL_MAP = [
    (r'\bshall\b', 'must'),
    (r'\bhereinafter\b', 'from this point on'),
    (r'\bpursuant to\b', 'under'),
    (r'\bnotwithstanding\b', 'despite'),
    (r'\bin the event that\b', 'if'),
    (r'\bprior to\b', 'before'),
    (r'\bsubsequent to\b', 'after'),
    (r'\bcommence\b', 'start'),
    (r'\bterminate\b', 'end'),
    (r'\butilize\b', 'use'),
    (r'\baforementioned\b', 'earlier mentioned'),
    (r'\bnull and void\b', 'not valid'),
    (r'\bcease and desist\b', 'stop'),
    (r'\bfull force and effect\b', 'still valid'),
]

def clean_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

def simplify_sentence(sentence):
    s = sentence.strip()
    if not s:
        return ""
    for pat, repl in LEGAL_MAP:
        s = re.sub(pat, repl, s, flags=re.I)
    s = re.sub(r'\bthe party of the first part\b', 'the first person or group', s, flags=re.I)
    s = re.sub(r'\bthe party of the second part\b', 'the second person or group', s, flags=re.I)
    s = clean_spaces(s)
    if len(s) > 140:
        parts = s.split(',')
        if len(parts) > 1:
            s = parts[0].strip() + ". " + ", ".join(p.strip() for p in parts[1:]).strip()
    return s

def split_sentences(text):
    return [x.strip() for x in re.split(r'(?<=[.!?])\s+', text) if x.strip()]

def summarize(text):
    sentences = split_sentences(text)
    simple = [simplify_sentence(s) for s in sentences]
    simple = [s for s in simple if s]
    if not simple:
        return "Paste legal text to see a simpler version."
    return " ".join(simple[:8])

def extract_points(text):
    points = []
    for sent in split_sentences(text):
        s = simplify_sentence(sent)
        if re.search(r'\b(pay|payment|rent|fee|deadline|due|terminate|end|cancel|refund|renew|liability|owe|must|agree|notice|penalty|late)\b', s, flags=re.I):
            points.append(s)
        elif len(s.split()) <= 16:
            points.append(s)
    seen = set()
    out = []
    for p in points:
        k = p.lower()
        if k not in seen:
            seen.add(k)
            out.append(p)
    return out[:7]


In [37]:
title = widgets.HTML(f"""
<div style="
    max-width: 900px;
    margin: 0 auto 18px auto;
    padding: 26px 22px;
    border-radius: 24px;
    background: linear-gradient(135deg, #0f172a, #2563eb);
    color: white;
    text-align: center;
    box-shadow: 0 14px 40px rgba(15,23,42,0.22);
">
  <div style="font-size: 42px; font-weight: 800; letter-spacing: 0.4px;">DocLens</div>
  <div style="font-size: 16px; opacity: 0.92; margin-top: 8px;">Turn legal text into clearer English.</div>
</div>
""")

subtitle = widgets.HTML("""
<div style="text-align:center; color:#475569; margin-bottom: 14px;">
  Simple, centered, and built for easier reading.
</div>
""")

input_box = widgets.Textarea(
    value='',
    placeholder='Paste a contract, lease, terms of service, or other legal text here...',
    layout=widgets.Layout(width='100%', height='240px')
)

button = widgets.Button(
    description='Simplify with DocLens',
    button_style='info',
    layout=widgets.Layout(width='240px', height='44px')
)

plans = widgets.HTML("""
<div style="
    max-width: 900px;
    margin: 18px auto 0 auto;
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 12px;
">
  <div style="background:white;border:1px solid #e5e7eb;border-radius:18px;padding:16px;text-align:center;box-shadow:0 4px 16px rgba(0,0,0,0.04);">
    <div style="font-weight:700;font-size:18px;">Free</div>
    <div style="color:#64748b;margin-top:8px;">1 document per day</div>
  </div>
  <div style="background:white;border:1px solid #c7d2fe;border-radius:18px;padding:16px;text-align:center;box-shadow:0 4px 16px rgba(0,0,0,0.04);">
    <div style="font-weight:700;font-size:18px;">Pro</div>
    <div style="color:#64748b;margin-top:8px;">Unlimited docs + saved history</div>
  </div>
  <div style="background:white;border:1px solid #bae6fd;border-radius:18px;padding:16px;text-align:center;box-shadow:0 4px 16px rgba(0,0,0,0.04);">
    <div style="font-weight:700;font-size:18px;">Business</div>
    <div style="color:#64748b;margin-top:8px;">Team tools + contract review</div>
  </div>
</div>
""")

output = widgets.Output()


In [38]:
title = widgets.HTML(f"""
<div style="
    max-width: 900px;
    margin: 0 auto 18px auto;
    padding: 26px 22px;
    border-radius: 24px;
    background: linear-gradient(135deg, #0f172a, #2563eb);
    color: white;
    text-align: center;
    box-shadow: 0 14px 40px rgba(15,23,42,0.22);
">
  <div style="font-size: 42px; font-weight: 800; letter-spacing: 0.4px;">DocLens</div>
  <div style="font-size: 16px; opacity: 0.92; margin-top: 8px;">Turn legal text into clearer English.</div>
</div>
""")

subtitle = widgets.HTML("""
<div style="text-align:center; color:#475569; margin-bottom: 14px;">
  Simple, centered, and built for easier reading.
</div>
""")
def on_click(b):
    with output:
        clear_output()
        doc = input_box.value.strip()

        if not doc:
            display(IPHTML("<div style='text-align:center;color:#b91c1c;font-weight:700;'>Paste some legal text first.</div>"))
            return

        summary = summarize(doc)
        points = extract_points(doc)

        display(IPHTML(f"""
        <div style="max-width: 900px; margin: 0 auto;">
          <div style="background:white;border:1px solid #e5e7eb;border-radius:20px;padding:18px;box-shadow:0 4px 16px rgba(0,0,0,0.04);margin-top:14px;text-align:center;">
            <h2 style="margin-top:0;color:#0f172a;">What it means</h2>
            <p style="line-height:1.9;font-size:16px;text-align:left;">{summary}</p>
          </div>
        </div>
        """))

        bullets = "".join([f"<li style='margin-bottom:8px;'>{p}</li>" for p in points]) if points else "<li>No clear key points found yet.</li>"
        display(IPHTML(f"""
        <div style="max-width: 900px; margin: 0 auto;">
          <div style="background:white;border:1px solid #e5e7eb;border-radius:20px;padding:18px;box-shadow:0 4px 16px rgba(0,0,0,0.04);margin-top:14px;text-align:center;">
            <h2 style="margin-top:0;color:#0f172a;">Important points</h2>
            <ul style="line-height:1.7;text-align:left;">{bullets}</ul>
          </div>
        </div>
        """))

        display(IPHTML("""
        <div style="max-width: 900px; margin: 14px auto 0 auto; padding: 12px 14px; border-radius: 14px; background: #fff7ed; border: 1px solid #fed7aa; color: #9a3412; font-size: 14px; text-align:center;">
          Not legal advice. This is for easier understanding only.
        </div>
        """))

button.on_click(on_click)
display(title, subtitle, widgets.VBox([input_box, widgets.HBox([button], layout=widgets.Layout(justify_content='center'))]), output, plans)


HTML(value='\n<div style="\n    max-width: 900px;\n    margin: 0 auto 18px auto;\n    padding: 26px 22px;\n   …

HTML(value='\n<div style="text-align:center; color:#475569; margin-bottom: 14px;">\n  Simple, centered, and bu…

Output()

HTML(value='\n<div style="\n    max-width: 900px;\n    margin: 18px auto 0 auto;\n    display: grid;\n    grid…